# Named Entity Recognition (NER)

Yeh notebook [AI for Beginners Curriculum](http://aka.ms/ai-beginners) se li gayi hai.

Is example mein hum seekhenge ki [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/datasets/abhinavwalia95/entity-annotated-corpus) Dataset se NER model ko kaise train karein. Aage badhne se pehle, kripya [ner_dataset.csv](https://www.kaggle.com/datasets/abhinavwalia95/entity-annotated-corpus?resource=download&select=ner_dataset.csv) file ko current directory mein download kar lijiye.

In [ ]:
import pandas as pd
from tensorflow import keras
import numpy as np

## Dataset taiyaar karna

Hum dataset ko ek data frame mein padhne se shuru karenge. Agar aap Pandas use karne ke baare mein aur jaanna chahte hain, toh hamare [Data Science for Beginners](http://aka.ms/datascience-beginners) mein [data processing par lesson](https://github.com/microsoft/Data-Science-For-Beginners/tree/main/2-Working-With-Data/07-python) dekhiye.

In [ ]:
df = pd.read_csv('ner_dataset.csv',encoding='unicode-escape')
df.head()

In [ ]:
tags = df.Tag.unique()
tags

In [ ]:
id2tag = dict(enumerate(tags))
tag2id = { v : k for k,v in id2tag.items() }

id2tag[0]

Ab humein vocabulary ke saath bhi yahi karna hoga. Simplicity ke liye, hum word frequency ko dhyan mein rakhe bina vocabulary banayenge; real life mein aap Keras vectorizer use karna chahenge aur words ki ginti ko limit karna chahenge.

In [ ]:
vocab = set(df['Word'].apply(lambda x: x.lower()))
id2word = { i+1 : v for i,v in enumerate(vocab) }
id2word[0] = '<UNK>'
vocab.add('<UNK>')
word2id = { v : k for k,v in id2word.items() }

Humein training ke liye sentences ka ek dataset banana hoga. Chalo original dataset par loop karte hain aur saare individual sentences ko `X` (words ki list) aur `Y` (tokens ki list) mein split karte hain:

In [ ]:
X,Y = [],[]
s,t = [],[]
for i,row in df[['Sentence #','Word','Tag']].iterrows():
    if pd.isna(row['Sentence #']):
        s.append(row['Word'])
        t.append(row['Tag'])
    else:
        if len(s)>0:
            X.append(s)
            Y.append(t)
        s,t = [row['Word']],[row['Tag']]
X.append(s)
Y.append(t)


In [ ]:
def vectorize(seq):
    return [word2id[x.lower()] for x in seq]

def tagify(seq):
    return [tag2id[x] for x in seq]

Xv = list(map(vectorize,X))
Yv = list(map(tagify,Y))

Xv[0], Yv[0]

Simplicity ke liye, hum saare sentences ko maximum length tak 0 tokens ke saath pad karenge. Real life mein hum zyada samajhdaar strategy use karna chahenge, aur sequences ko sirf ek minibatch ke andar pad karenge.

In [ ]:
X_data = keras.preprocessing.sequence.pad_sequences(Xv,padding='post')
Y_data = keras.preprocessing.sequence.pad_sequences(Yv,padding='post')

## Token classification network define karna

Hum token classification ke liye do-layer bidirectional LSTM network use karenge. Aakhri LSTM layer ke har output par dense classifier apply karne ke liye, hum `TimeDistributed` structure use karenge, jo LSTM ke har step par uske har output ke liye wahi ek dense layer repeat karta hai:

In [ ]:
maxlen = X_data.shape[1]
vocab_size = len(vocab)
num_tags = len(tags)
model = keras.models.Sequential([
    keras.layers.Embedding(vocab_size, 300, input_length=maxlen),
    keras.layers.Bidirectional(keras.layers.LSTM(units=100, activation='tanh', return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(units=100, activation='tanh', return_sequences=True)),
    keras.layers.TimeDistributed(keras.layers.Dense(num_tags, activation='softmax'))
])
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['acc'])
model.summary()

Yahan dhyan dijiye ki hum apne dataset ke liye `maxlen` ko explicitly specify kar rahe hain - agar hum chahte hain ki network variable length wali sequences ko handle kar sake, toh network define karte waqt humein thoda aur chalaak hona padega.

Ab chalo model ko train karte hain. Speed ke liye hum sirf ek epoch ke liye train karenge, lekin aap ise zyada der tak train karne ki koshish kar sakte hain. Saath hi, aap dataset ka kuch hissa training dataset se alag rakhna chahenge, taaki validation accuracy observe ki ja sake.

In [ ]:
model.fit(X_data,Y_data)

## Result ko test karna

Ab dekhte hain ki hamara entity recognition model ek sample sentence par kaise kaam karta hai:

In [ ]:
sent = 'John Smith went to Paris to attend a conference in cancer development institute'
words = sent.lower().split()
v = keras.preprocessing.sequence.pad_sequences([[word2id[x] for x in words]],padding='post',maxlen=maxlen)
res = model(v)[0]

In [ ]:
r = np.argmax(res.numpy(),axis=1)
for i,w in zip(r,words):
    print(f"{w} -> {id2tag[i]}")

## Key takeaways

Ek simple LSTM model bhi NER mein reasonable results dikhata hai. Lekin behtar results paane ke liye, aap BERT jaise bade pre-trained language models use karna chahenge. Huggingface Transformers library use karke NER ke liye BERT ko train karne ki process [yahan](https://huggingface.co/course/chapter7/2?fw=pt) describe ki gayi hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->